In [12]:
##### Takes raw scenario from Jung et al (2021) and 
# scenario data can be downloaded from: https://doi.org/10.1038/s41559-021-01569-y 

import pandas as pd
import geopandas as gpd
import numpy as np
import rasterio
from rasterio.warp import reproject
from rasterio.enums import Resampling
from pathlib import Path
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch

In [13]:
##### Load data

# Get the current working directory
cd = Path.cwd().parent 

# priority rasters
biodiversity_with_PA = f"{cd}/Data/minshort_speciestargets_biome.id_withPA_esh10km_repruns10_ranked.tif"

# set directories
DATA_DIR = Path(f"{cd}/Data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

RESULTS_DIR = Path(f"{cd}/Results/30x30")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)


In [14]:
##### Create raster identifying top 30%
# 0 = no in top 30%, 1 = in top 30%, not PA, 2 = in top 30%, already PA

# define function to reclassify
def reclassify_PA_raster(in_path):
    with rasterio.open(in_path) as src:
        data = src.read(1)
        nodata = src.nodata
        profile = src.profile.copy()

    reclass = np.full_like(data, fill_value=0 if nodata is None else nodata)

    # Apply reclassification rules
    reclass = np.where((data >= 16) & (data <= 85), 0, reclass)
    reclass = np.where((data >= 86) & (data <= 100), 1, reclass)
    reclass = np.where((data >= 0) & (data <= 15), 2, reclass)

    # Preserve nodata 
    if nodata is not None:
        reclass = np.where(data == nodata, nodata, reclass)

    profile.update(dtype=reclass.dtype, nodata=nodata)

    return reclass, profile, nodata

def save_raster(reclass, profile, out_path):
    with rasterio.open(out_path, "w", **profile) as dst:
        dst.write(reclass, 1)

# run
biodiversity_reclass, biodiversity_profile, _ = reclassify_PA_raster(biodiversity_with_PA)
save_raster(biodiversity_reclass, biodiversity_profile, RESULTS_DIR / "conservation_scenario.tif" )

# save
with rasterio.open(RESULTS_DIR / "conservation_scenario.tif", "w", **biodiversity_profile) as dst:
        dst.write(biodiversity_reclass, 1)